# BipedalWalker-v3 + SAC
### Project 5 — Reinforcement Learning in Continuous Spaces

| | |
|---|---|
| **Environment** | `BipedalWalker-v3` (Gymnasium / Box2D) |
| **Algorithm** | SAC — Soft Actor-Critic (stable-baselines3) |
| **Goal** | 8 pts: hyperparameter sweep + architecture comparison + deterministic evaluation |

Run cells top-to-bottom. Training is the slow part — kick it off overnight.

## 0. Installation

In [1]:
# Run once, then restart kernel
# !pip install stable-baselines3[extra] gymnasium[box2d] torch matplotlib numpy

## 1. Imports & Sanity Check

In [2]:
import os, sys, json
import numpy as np
import torch
import gymnasium as gym

print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {torch.cuda.is_available()} | device = {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Gymnasium  : {gym.__version__}")

env = gym.make("BipedalWalker-v3")
obs, _ = env.reset()
print(f"\nObservation space : {env.observation_space}  →  shape {obs.shape}")
print(f"Action space      : {env.action_space}")
env.close()

Python     : 3.12.0
PyTorch    : 2.12.0+cu130
CUDA       : True | device = NVIDIA GeForce RTX 3070
Gymnasium  : 1.2.3

Observation space : Box([-3.1415927 -5.        -5.        -5.        -3.1415927 -5.
 -3.1415927 -5.        -0.        -3.1415927 -5.        -3.1415927
 -5.        -0.        -1.        -1.        -1.        -1.
 -1.        -1.        -1.        -1.        -1.        -1.       ], [3.1415927 5.        5.        5.        3.1415927 5.        3.1415927
 5.        5.        3.1415927 5.        3.1415927 5.        5.
 1.        1.        1.        1.        1.        1.        1.
 1.        1.        1.       ], (24,), float32)  →  shape (24,)
Action space      : Box(-1.0, 1.0, (4,), float32)


/home/mitchellds/studia/sem6/IOwADC/.venv2/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## 2. Configuration

In [3]:
from config import HYPERPARAMS, ARCHITECTURES, TOTAL_TIMESTEPS, N_SEEDS, DEVICE

print(f"Device          : {DEVICE}")
print(f"Timesteps/run   : {TOTAL_TIMESTEPS:,}")
print(f"Seeds/config    : {N_SEEDS}")
print(f"Total runs      : {len(HYPERPARAMS) * len(ARCHITECTURES) * N_SEEDS}")

print(f"\n── Hyperparameter sets ({len(HYPERPARAMS)}) " + "─"*40)
for name, hp in HYPERPARAMS.items():
    print(f"  {name}")
    for k, v in hp.items():
        print(f"    {k:<20} = {v}")

print(f"\n── Architectures ({len(ARCHITECTURES)}) " + "─"*44)
for name, arch in ARCHITECTURES.items():
    print(f"  {name}:  net_arch={arch['net_arch']},  activation={arch['activation_fn'].__name__}")

Device          : cuda
Timesteps/run   : 400,000
Seeds/config    : 5
Total runs      : 30

── Hyperparameter sets (3) ────────────────────────────────────────
  hp_default
    learning_rate        = 0.0003
    batch_size           = 256
    tau                  = 0.005
    gamma                = 0.99
    learning_starts      = 10000
    buffer_size          = 300000
    ent_coef             = auto
  hp_high_lr
    learning_rate        = 0.001
    batch_size           = 256
    tau                  = 0.005
    gamma                = 0.99
    learning_starts      = 10000
    buffer_size          = 300000
    ent_coef             = auto
  hp_large_batch
    learning_rate        = 0.0003
    batch_size           = 512
    tau                  = 0.02
    gamma                = 0.995
    learning_starts      = 10000
    buffer_size          = 300000
    ent_coef             = auto

── Architectures (2) ────────────────────────────────────────────
  arch_small:  net_arch=[64, 64],  activation

## 3. Environment Benchmark

Measure simulation speed on your hardware.
Results are saved to `models/benchmark.json`.

In [3]:
from train import run_benchmark
run_benchmark()

Benchmarking environment step time...
  Steps/sec    : 1701.4
  ms/step      : 0.588
  Episodes done: 3


## 4. Training

**Full sweep:** `3 HP sets × 2 architectures × 10 seeds = 60 runs`

Recommended: run `python train.py` in a terminal overnight.  
Results are saved incrementally to `results/*.csv` — you can plot mid-training.

In [ ]:
# ── Option A: full sweep directly in notebook (blocking) ──────────────────
# from train import train_all
# train_all(total_timesteps=TOTAL_TIMESTEPS, resume=False)

# ── Option B: single config — good for testing ────────────────────────────
# from train import train_single
# train_single("hp_default", "arch_large", seed=0, total_timesteps=50_000)

# ── Option C: shell out ───────────────────────────────────────────────────
# !python train.py

### 4b. Resume (if training was interrupted)

In [ ]:
# Resume all unfinished runs:
# from train import train_all
# train_all(total_timesteps=TOTAL_TIMESTEPS, resume=True)

# Resume a specific run:
# from train import train_single
# train_single("hp_default", "arch_large", seed=3, total_timesteps=TOTAL_TIMESTEPS, resume=True)

## 5. Training Statistics

Summary of completed runs — final mean reward ± std across seeds.

In [4]:
from utils import load_results

header = f"{'Config':<42} {'Seeds':>6}  {'Final mean':>12}  {'± std':>8}"
print(header)
print("─" * len(header))

for hp_name in HYPERPARAMS:
    for arch_name in ARCHITECTURES:
        data = load_results("results", hp_name, arch_name)
        cfg = f"{hp_name}__{arch_name}"
        if not data:
            print(f"  {cfg:<40} {'—':>6}")
            continue
        print(f"  {cfg:<40} {data['n_seeds']:>6}  {data['mean'][-1]:>+12.2f}  {data['std'][-1]:>8.2f}")

Config                                      Seeds    Final mean     ± std
─────────────────────────────────────────────────────────────────────────
  hp_default__arch_small                        5        -48.87     41.73
  hp_default__arch_large                        5       +200.78    107.57
  hp_high_lr__arch_small                        5        -52.65     25.61
  hp_high_lr__arch_large                        5       +195.27    126.45
  hp_large_batch__arch_small                    5        -17.93     59.91
  hp_large_batch__arch_large                    5       +268.94      0.00


## 6. Learning Curves

- **Hyperparameter comparison** — 3 HP sets per architecture  
- **Architecture comparison** — 2 architectures per HP set  

Shaded area = ±1 std across 10 seeds.

In [5]:
from plot import plot_hyperparams_comparison, plot_arch_comparison
from IPython.display import Image, display

plot_hyperparams_comparison(show=False)
plot_arch_comparison(show=False)

for fname in ["hyperparams_comparison.png", "arch_comparison.png"]:
    path = os.path.join("plots", fname)
    if os.path.exists(path):
        display(Image(path))

ValueError: x and y must have same first dimension, but have shapes (40,) and (80,)

## 7. Deterministic Evaluation

Load saved agents, disable exploration (`deterministic=True`), collect rewards.  
Compares against the stochastic training curve.

In [6]:
from evaluate import find_best_model

best = find_best_model(n_episodes=20)


── hp_default / arch_small ──────────────────────────────────


KeyboardInterrupt: 

### 7b. Eval Overlay Plot

In [ ]:
from plot import plot_eval_overlay
from IPython.display import Image, display

plot_eval_overlay(show=False)

path = "plots/eval_overlay.png"
if os.path.exists(path):
    display(Image(path))

## 8. Render Best Agent

Watch the trained agent walk. Requires a local display — won't work on a headless server.

In [ ]:
from evaluate import evaluate_single

best_meta_path = "models/best_config.json"
if os.path.exists(best_meta_path):
    with open(best_meta_path) as f:
        meta = json.load(f)
    hp_name, arch_name = meta["config"].split("__", 1)
    best_seed = max(meta["per_seed"], key=lambda x: x["mean_reward"])["seed"]
    print(f"Best config : {meta['config']}")
    print(f"Best seed   : {best_seed}  (mean reward = {max(meta['per_seed'], key=lambda x: x['mean_reward'])['mean_reward']:.2f})")
    evaluate_single(hp_name, arch_name, best_seed, n_episodes=3, render=True)
else:
    print("Run cell 7 first (find_best_model).")